# Research Intelligence Platform — Step-by-Step Test Notebook

Run each section **in order**. Every cell prints its own pass/fail result so you can stop the moment something breaks.

| Section | What it tests |
|---|---|
| 0 — Setup | env, Neo4j connection, schema |
| 1 — Crawler | arXiv RSS fetch + store |
| 2 — Phase 1: Enrichment | Semantic Scholar, OpenAlex, rank_score, CITES edges |
| 3 — Phase 2: Embeddings | SPECTER2 paper-level embed, vector index, semantic search |
| 4 — Phase 3: Clustering | UMAP + HDBSCAN, LLM topic naming, trend scores |
| 5 — Phase 4: Citation Flow | reverse traversal, divergence, ASCII + JSON render |
| 6 — CLI smoke test | run every `kg` command and check exit codes |


---
## 0 — Setup
Check Python path, load .env, verify Neo4j is reachable, and ensure the schema exists.

In [ ]:
# ── 0-A: path & env ──────────────────────────────────────────────────────────
import sys, os
from pathlib import Path

# Make sure the repo root is on sys.path so `import kg` works
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root : {REPO_ROOT}")
print(f"Python    : {sys.version.split()[0]}")

# Load .env
from dotenv import load_dotenv
loaded = load_dotenv(REPO_ROOT / ".env")
print(f".env loaded : {loaded}")

from kg.utils.config import get_settings
cfg = get_settings()
print(f"Neo4j URI  : {cfg.neo4j_uri}")
print(f"Together   : {'SET' if cfg.together_api_key else 'NOT SET — LLM steps will be skipped'}")
print(f"S2 API key : {'SET' if cfg.semantic_scholar_api_key else 'not set (1 req/s limit)'}")
print("✅ Setup OK")

In [ ]:
# ── 0-B: Neo4j connection ─────────────────────────────────────────────────────
from kg.graph.neo4j_client import Neo4jClient

db = Neo4jClient()
try:
    db.connect()
    count = db.get_paper_count()
    print(f"✅ Neo4j connected — {count} Paper nodes in graph")
    db.close()
except Exception as e:
    print(f"❌ Neo4j connection FAILED: {e}")
    print("   → Is Docker running? Try: docker compose up -d")

In [ ]:
# ── 0-C: schema setup (idempotent — safe to re-run) ──────────────────────────
db = Neo4jClient()
db.connect()
db.setup_schema()
db.close()

# Verify vector index exists
db = Neo4jClient()
db.connect()
indexes = db.run_query("SHOW INDEXES YIELD name, type WHERE name = 'paper_embedding' RETURN name, type")
db.close()

if indexes:
    print(f"✅ Schema OK — paper_embedding vector index: {indexes[0]['type']}")
else:
    print("⚠️  paper_embedding index not found — Neo4j version may not support vector indexes")

---
## 1 — arXiv Crawler
Fetches papers from all 9 RSS feeds and writes them to Neo4j.

> **⚠️ Weekend behaviour:** arXiv publishes no new papers on Saturday or Sunday.
> The RSS feeds contain a `<skipDays>` tag on weekends and return 0 entries — this is **expected and normal**.
> Run this on a weekday to get real papers. The rest of the notebook (enrichment, embeddings, flow) works
> on any papers already in Neo4j, so you can continue even if the crawler returns 0 today.
>
> **Tip:** `max_papers_per_feed=5` keeps this fast during testing. Set to 100+ for a real run.

In [ ]:
# ── 1-A: fetch feeds (no database write yet) ─────────────────────────────────
# NOTE: feeds use https:// — the old http:// URLs caused a 301 redirect that
# sometimes returned an empty body. Fixed in kg/crawlers/arxiv.py.
import asyncio
from datetime import datetime
from kg.crawlers.arxiv import ArxivCrawler, ARXIV_FEEDS

today = datetime.now()
weekday_name = today.strftime("%A")
is_weekend = today.weekday() >= 5   # 5=Sat, 6=Sun

print(f"Today is {weekday_name} — {'⚠️  WEEKEND: arXiv feeds will be empty (expected)' if is_weekend else '✅ weekday: feeds should have papers'}")
print(f"\nFeeds configured: {len(ARXIV_FEEDS)}")
for f in ARXIV_FEEDS:
    print(f"  {f}")
print()

crawler = ArxivCrawler(max_papers_per_feed=5)
raw = await crawler.fetch()

if len(raw) == 0 and is_weekend:
    print(f"\n✅ Raw entries fetched: 0  (expected on weekends — arXiv skips Sat/Sun)")
    print(   "   The cells below (1-B, 1-C) will store 0 papers — that's fine.")
    print(   "   Sections 2–6 still work on papers already in Neo4j.")
elif len(raw) == 0:
    print(f"\n❌ Raw entries fetched: 0 on a weekday — unexpected")
    print(   "   → Check internet access or try: curl https://export.arxiv.org/rss/cs.LG")
else:
    print(f"\n✅ Raw entries fetched: {len(raw)}")

In [ ]:
# ── 1-B: parse entries ────────────────────────────────────────────────────────
parsed = await crawler.parse(raw)
print(f"Parsed papers: {len(parsed)}")

if parsed:
    sample = parsed[0]
    print(f"\nSample paper:")
    print(f"  arxiv_id  : {sample['arxiv_id']}")
    print(f"  title     : {sample['title'][:70]}")
    print(f"  authors   : {sample['authors'][:3]}")
    print(f"  date      : {sample['published_date']}")
    print(f"  categories: {sample['categories'][:3]}")
    print("✅ Parse working")
else:
    print("ℹ️  0 papers parsed — OK if running on a weekend (see note above)")

In [ ]:
# ── 1-C: store to Neo4j ───────────────────────────────────────────────────────
await crawler.store(parsed)

db = Neo4jClient()
db.connect()
count_after = db.get_paper_count()
author_count = db.run_query("MATCH (a:Author) RETURN count(a) AS n")[0]["n"]
db.close()

print(f"Neo4j — Papers: {count_after}  |  Authors: {author_count}")

if count_after > 0:
    print("✅ Papers available in graph — continuing to next sections")
else:
    print("⚠️  Graph is empty — sections 2–6 need at least 1 paper. Run crawler on a weekday.")

In [ ]:
# ── 1-D: keyword search (sanity check) ───────────────────────────────────────
db = Neo4jClient()
db.connect()
results = db.search_papers("transformer", limit=5)
db.close()

print(f"Keyword search 'transformer': {len(results)} results")
for r in results:
    print(f"  [{r.get('published_date','?')[:10]}] {r['title'][:65]}")

if results:
    print("✅ Keyword search working")
else:
    print("ℹ️  No results — graph may be empty or no transformer papers crawled yet")

---
## 2 — Phase 1: Citation Enrichment
Tests Semantic Scholar, OpenAlex, rank_score computation, CITES edges, and Institution nodes.

> Uses real API calls — 1 paper max to stay within rate limits during testing.

In [ ]:
# ── 2-A: Semantic Scholar — fetch a known paper ──────────────────────────────
# Using "Attention Is All You Need" (2017) — very well indexed in S2
from kg.enrichment.semantic_scholar import fetch_paper as s2_fetch

TEST_ARXIV_ID = "1706.03762"   # Attention Is All You Need

s2_data = s2_fetch(TEST_ARXIV_ID)

if s2_data:
    print(f"✅ Semantic Scholar data for {TEST_ARXIV_ID}:")
    print(f"   citation_count             : {s2_data['citation_count']}")
    print(f"   influential_citation_count : {s2_data['influential_citation_count']}")
    print(f"   year                       : {s2_data['year']}")
    print(f"   venue                      : {s2_data['venue']}")
    print(f"   references (arXiv IDs)     : {s2_data['references'][:5]}  ... ({len(s2_data['references'])} total)")
else:
    print(f"❌ Semantic Scholar returned None for {TEST_ARXIV_ID}")
    print("   → Check internet access / rate limit")

In [ ]:
# ── 2-B: OpenAlex — fetch the same paper ─────────────────────────────────────
from kg.enrichment.openalex import fetch_paper as oa_fetch

oa_data = oa_fetch(TEST_ARXIV_ID)

if oa_data:
    print(f"✅ OpenAlex data for {TEST_ARXIV_ID}:")
    print(f"   citation_velocity : {oa_data['citation_velocity']}")
    print(f"   authors ({len(oa_data['authors'])} total):")
    for a in oa_data["authors"][:3]:
        print(f"     {a['name']:<30}  h_index={a['h_index']}  institution={a['institution_name'][:40]}")
else:
    print(f"❌ OpenAlex returned None for {TEST_ARXIV_ID}")
    print("   → Check internet access")

In [ ]:
# ── 2-C: rank_score formula ───────────────────────────────────────────────────
import math
from kg.enrichment.runner import _rank_score, _recency_score

# Compute with real data if available, else use mock
citation_count    = (s2_data or {}).get("citation_count", 100)
citation_velocity = (oa_data or {}).get("citation_velocity", 50.0)
published_date    = "2017-06-12"   # Attention Is All You Need
authors           = (oa_data or {}).get("authors", [])
author_influence  = max((a.get("h_index", 0) or 0 for a in authors), default=0)
recency           = _recency_score(published_date)

rank = _rank_score(citation_count, citation_velocity, recency, author_influence)

print("rank_score formula:  log(cit+1) + 1.5*vel + 0.8*recency + 0.3*author_influence")
print(f"  log({citation_count}+1)          = {math.log(citation_count+1):.4f}")
print(f"  1.5 × {citation_velocity:.2f}   = {1.5*citation_velocity:.4f}")
print(f"  0.8 × {recency:.4f} (recency) = {0.8*recency:.4f}")
print(f"  0.3 × {author_influence} (h_index)  = {0.3*author_influence:.4f}")
print(f"  ─────────────────────────────────")
print(f"  rank_score                = {rank:.4f}")
print(f"\n✅ rank_score formula working")

In [ ]:
# ── 2-D: LLM judge trigger logic ─────────────────────────────────────────────
from kg.enrichment.llm_judge import should_judge

# Case 1: both APIs returned data — should NOT trigger judge
assert not should_judge(s2_data or {"citation_count": 1}, oa_data or {"citation_velocity": 1.0}, cited_in_graph=False), \
    "should_judge returned True when data is present — unexpected"

# Case 2: both APIs empty — SHOULD trigger judge
assert should_judge(None, None, cited_in_graph=False), \
    "should_judge returned False when both APIs are None — unexpected"

# Case 3: citation_count=0 but paper IS cited by others — SHOULD trigger judge
assert should_judge({"citation_count": 0, "influential_citation_count": 0, "references": []}, None, cited_in_graph=True), \
    "should_judge returned False for 0-citation paper that appears in other papers' refs"

print("✅ LLM judge trigger logic: all 3 cases correct")

In [ ]:
# ── 2-E: run enrichment pipeline on real graph (dry_run=True — no writes) ────
from kg.enrichment.runner import run_enrichment

print("Running enrichment pipeline (dry_run=True — no DB writes)...")
stats = run_enrichment(limit=3, dry_run=True)
print(f"\n✅ Enrichment runner completed")
print(f"   total={stats.total}  s2_found={stats.s2_found}  oa_found={stats.oa_found}  failed={stats.failed}")

In [ ]:
# ── 2-F: run enrichment pipeline for real (writes rank_score, CITES, Institutions) ─
# Set limit=5 for testing — increase for production runs
print("Running enrichment pipeline (LIVE — writing to Neo4j)...")
stats = run_enrichment(limit=5, dry_run=False)

# Verify rank_score was written
db = Neo4jClient()
db.connect()
enriched_papers = db.run_query("""
    MATCH (p:Paper) WHERE p.rank_score IS NOT NULL
    RETURN p.arxiv_id AS id, p.rank_score AS score, p.citation_count AS cit
    ORDER BY p.rank_score DESC LIMIT 5
""")
cites_count = db.run_query("MATCH ()-[:CITES]->() RETURN count(*) AS n")[0]["n"]
inst_count  = db.run_query("MATCH (i:Institution) RETURN count(i) AS n")[0]["n"]
db.close()

print(f"\nTop papers by rank_score:")
for r in enriched_papers:
    print(f"  {r['id']}  score={r['score']:.3f}  citations={r['cit']}")

print(f"\nCITES edges : {cites_count}")
print(f"Institutions: {inst_count}")

if enriched_papers:
    print("\n✅ Phase 1 enrichment working")
else:
    print("\n⚠️  No enriched papers found — papers may already be enriched or APIs unreachable")

---
## 3 — Phase 2: SPECTER2 Embeddings + Vector Search

> ⚠️ First run downloads the SPECTER2 model (~440 MB). Subsequent runs are fast.
>
> Requires: `pip install transformers adapters torch`

In [ ]:
# ── 3-A: load SPECTER2 and embed two test papers ─────────────────────────────
# NOTE: importing embedder triggers model download on first run
print("Loading SPECTER2 (this may take a minute on first run)...")
from kg.nlp.embedder import embed_papers, embed_query, cosine_similarity

test_papers = [
    {"title": "Attention Is All You Need",
     "abstract": "We propose a new architecture, the Transformer, based solely on attention mechanisms."},
    {"title": "LoRA: Low-Rank Adaptation of Large Language Models",
     "abstract": "We propose Low-Rank Adaptation, which freezes pretrained weights and injects trainable rank decomposition matrices."},
    {"title": "Deep Residual Learning for Image Recognition",
     "abstract": "We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously."},
]

embeddings = embed_papers(test_papers)

for p, emb in zip(test_papers, embeddings):
    status = f"dim={len(emb)}" if emb else "FAILED"
    print(f"  {p['title'][:50]:<52} → {status}")

if all(e is not None for e in embeddings):
    print("\n✅ SPECTER2 embeddings generated")
else:
    print("\n❌ Some embeddings failed — check torch/transformers install")

In [ ]:
# ── 3-B: cosine similarity between papers ─────────────────────────────────────
if all(e is not None for e in embeddings):
    sim_attn_lora = cosine_similarity(embeddings[0], embeddings[1])
    sim_attn_resnet = cosine_similarity(embeddings[0], embeddings[2])
    sim_lora_resnet = cosine_similarity(embeddings[1], embeddings[2])

    print("Cosine similarity matrix (SPECTER2):")
    print(f"  Attention vs LoRA     : {sim_attn_lora:.4f}  (expect HIGH — both are transformer/LLM papers)")
    print(f"  Attention vs ResNet   : {sim_attn_resnet:.4f}  (expect MEDIUM — both deep learning but different domain)")
    print(f"  LoRA vs ResNet        : {sim_lora_resnet:.4f}  (expect LOWER — LoRA is LLM specific)")

    # Attention and LoRA should be more similar to each other than to ResNet
    if sim_attn_lora > sim_attn_resnet:
        print("\n✅ Similarity ordering correct — citation-space proximity works")
    else:
        print("\n⚠️  Unexpected ordering — may be OK if abstracts are too short to capture full signal")
else:
    print("⚠️  Skipped — embeddings not available")

In [ ]:
# ── 3-C: embed_query (free-text semantic search input) ───────────────────────
query_vec = embed_query("efficient fine-tuning of large language models")

if query_vec:
    print(f"✅ embed_query returned vector dim={len(query_vec)}")
    # Show similarity to our 3 test papers
    for p, emb in zip(test_papers, embeddings):
        if emb:
            sim = cosine_similarity(query_vec, emb)
            print(f"  query vs '{p['title'][:45]:<47}': {sim:.4f}")
else:
    print("❌ embed_query failed")

In [ ]:
# ── 3-D: write embeddings to Neo4j + verify ──────────────────────────────────
print("Running paper-level embedding pipeline (limit=10)...")
from kg.nlp.embedder import run_embedding_pipeline

result = run_embedding_pipeline(limit=10, dry_run=False)
print(f"Embedded: {result['papers_embedded']}  |  Failed: {result['failed']}")

# Verify embedding was stored
db = Neo4jClient()
db.connect()
with_emb = db.run_query("MATCH (p:Paper) WHERE p.embedding IS NOT NULL RETURN count(p) AS n")[0]["n"]
without_emb = db.run_query("MATCH (p:Paper) WHERE p.embedding IS NULL RETURN count(p) AS n")[0]["n"]
db.close()

print(f"\nPapers with embedding   : {with_emb}")
print(f"Papers without embedding: {without_emb}")

if with_emb > 0:
    print("✅ Embeddings stored in Neo4j")
else:
    print("⚠️  No embeddings stored — check SPECTER2 install or run crawler first")

In [ ]:
# ── 3-E: vector search (semantic search via Neo4j vector index) ───────────────
if query_vec and with_emb > 0:
    db = Neo4jClient()
    db.connect()
    semantic_results = db.vector_search_papers(query_vec, top_k=5)
    db.close()

    print("Semantic search: 'efficient fine-tuning of large language models'")
    if semantic_results:
        for r in semantic_results:
            print(f"  [{r.get('score', 0):.4f}] {r.get('title','?')[:65]}")
        print("\n✅ Vector search working")
    else:
        print("⚠️  No vector search results — verify Neo4j vector index is populated")
else:
    print("⚠️  Skipped — embed query or stored embeddings not available")

---
## 4 — Phase 3: Topic Clustering + Trend Detection

> Requires at least ~10 papers with embeddings for HDBSCAN to form clusters.
>
> Requires: `pip install umap-learn hdbscan`

In [ ]:
# ── 4-A: check we have enough embeddings for clustering ───────────────────────
db = Neo4jClient()
db.connect()
emb_count = db.run_query("MATCH (p:Paper) WHERE p.embedding IS NOT NULL RETURN count(p) AS n")[0]["n"]
db.close()

print(f"Papers with embeddings: {emb_count}")
if emb_count < 10:
    print("⚠️  Fewer than 10 embeddings — HDBSCAN needs ~10+ to form clusters")
    print("   → Run cell 3-D with a higher limit first (e.g. limit=100)")
    SKIP_CLUSTERING = True
else:
    SKIP_CLUSTERING = False
    print(f"✅ Sufficient embeddings for clustering ({emb_count} papers)")

In [ ]:
# ── 4-B: run UMAP + HDBSCAN (dry_run=True — inspect results before writing) ──
if SKIP_CLUSTERING:
    print("⚠️  Skipped — not enough embeddings")
else:
    from kg.clustering.cluster import run_clustering

    print("Running UMAP (768d→15d) + HDBSCAN (dry_run=True)...")
    result = run_clustering(min_cluster_size=5, dry_run=True)

    if result:
        print(f"\n✅ Clustering complete:")
        print(f"   Papers     : {result['n_papers']}")
        print(f"   Clusters   : {result['n_clusters']}")
        print(f"   Noise pts  : {result['n_noise']}")
        print(f"\n   Cluster names (LLM-generated):")
        for cid, name in result.get('cluster_names', {}).items():
            count = len([l for l in result['labels'] if l == cid])
            print(f"     [{cid}] {name}  ({count} papers)")
    else:
        print("❌ Clustering returned empty result")

In [ ]:
# ── 4-C: run clustering LIVE (writes Topic nodes + BELONGS_TO edges) ─────────
if SKIP_CLUSTERING:
    print("⚠️  Skipped")
else:
    from kg.clustering.cluster import run_clustering

    print("Running LIVE clustering (writing to Neo4j)...")
    result = run_clustering(min_cluster_size=5, dry_run=False)

    # Verify Topic nodes and BELONGS_TO edges
    db = Neo4jClient()
    db.connect()
    topics     = db.run_query("MATCH (t:Topic) RETURN t.name AS name, t.trend_score AS ts, t.paper_count AS pc ORDER BY t.trend_score DESC LIMIT 10")
    bt_edges   = db.run_query("MATCH ()-[:BELONGS_TO]->() RETURN count(*) AS n")[0]["n"]
    db.close()

    print(f"\nTopic nodes: {len(topics)}  |  BELONGS_TO edges: {bt_edges}")
    print("\nTop topics by trend_score:")
    for t in topics:
        arrow = "↑↑↑" if t["ts"] > 10 else ("↑↑" if t["ts"] > 3 else "↑")
        print(f"  {t['name']:<40} {arrow:4}  papers={t['pc']}  trend={t['ts']:.2f}")

    if topics:
        print("\n✅ Phase 3 clustering written to Neo4j")
    else:
        print("\n⚠️  No topic nodes found after write")

In [ ]:
# ── 4-D: trend score formula verification ────────────────────────────────────
# trend_score = new_papers_in_topic × avg_citation_velocity

new_papers_in_topic = 5
avg_velocity = 2.3
expected_trend = new_papers_in_topic * avg_velocity

print(f"trend_score formula: new_papers × avg_citation_velocity")
print(f"  {new_papers_in_topic} new papers × {avg_velocity} avg velocity = {expected_trend}")
print("\n✅ Trend formula verified")

---
## 5 — Phase 4: Citation Flow + Research River

Tests reverse traversal, divergence detection, ASCII rendering, and JSON export.

> Requires CITES edges in the graph — run Phase 1 enrichment first (cell 2-F).

In [ ]:
# ── 5-A: check CITES edges exist ──────────────────────────────────────────────
db = Neo4jClient()
db.connect()
cites_count = db.run_query("MATCH ()-[:CITES]->() RETURN count(*) AS n")[0]["n"]

# Pick the most-cited paper as our test target
most_cited = db.run_query("""
    MATCH (p:Paper)<-[:CITES]-()
    RETURN p.arxiv_id AS id, p.title AS title, count(*) AS cited_by
    ORDER BY cited_by DESC LIMIT 1
""")

# Also try finding a paper that cites others
has_refs = db.run_query("""
    MATCH (p:Paper)-[:CITES]->()
    RETURN p.arxiv_id AS id, p.title AS title, count(*) AS refs
    ORDER BY refs DESC LIMIT 1
""")

db.close()

print(f"CITES edges in graph: {cites_count}")

if most_cited:
    mc = most_cited[0]
    print(f"Most-cited paper : {mc['id']} — {mc['title'][:55]} (cited {mc['cited_by']} times)")

if has_refs:
    hr = has_refs[0]
    FLOW_TARGET = hr["id"]
    print(f"Best flow target : {hr['id']} — {hr['title'][:55]} ({hr['refs']} refs)")
    SKIP_FLOW = False
else:
    print("⚠️  No CITES edges found — run Phase 1 enrichment first")
    SKIP_FLOW = True
    FLOW_TARGET = None

In [ ]:
# ── 5-B: build citation tree ──────────────────────────────────────────────────
if SKIP_FLOW:
    print("⚠️  Skipped — no CITES edges")
else:
    from kg.flow.citation_flow import build_citation_tree, detect_divergence, render_tree, export_flow_json

    print(f"Building citation tree for: {FLOW_TARGET}  (depth=2)")
    tree = build_citation_tree(FLOW_TARGET, depth=2)

    if tree:
        def count_nodes(node):
            return 1 + sum(count_nodes(c) for c in node.children)

        total = count_nodes(tree)
        print(f"✅ Tree built — {total} nodes (root + ancestors)")
        print(f"   Root: {tree.arxiv_id} ({tree.published_date[:4]})")
        print(f"   Direct refs: {len(tree.children)}")
    else:
        print(f"❌ build_citation_tree returned None for {FLOW_TARGET}")

In [ ]:
# ── 5-C: divergence detection ─────────────────────────────────────────────────
if SKIP_FLOW or not tree:
    print("⚠️  Skipped")
else:
    detect_divergence(tree)

    # Count divergence annotations
    def count_divergent(node):
        n = 1 if node.divergence_score is not None else 0
        return n + sum(count_divergent(c) for c in node.children)

    divs = count_divergent(tree)
    print(f"Divergence annotations: {divs} nodes marked")
    print(f"(threshold = {0.6} — below this cosine = divergence)")

    if tree.children and len(tree.children) >= 2 and embeddings:
        print("\n✅ Divergence detection ran (score depends on embedding similarity)")
    else:
        print("✅ Divergence detection ran (0 divergences expected if <2 children or no embeddings)")

In [ ]:
# ── 5-D: ASCII terminal render ────────────────────────────────────────────────
if SKIP_FLOW or not tree:
    print("⚠️  Skipped")
else:
    ascii_output = render_tree(tree)
    print("Citation ancestry tree (ASCII):")
    print("─" * 70)
    print(ascii_output)
    print("─" * 70)
    print("✅ ASCII render working")

In [ ]:
# ── 5-E: JSON export (Research River input) ───────────────────────────────────
import json

if SKIP_FLOW or not tree:
    print("⚠️  Skipped")
else:
    flow_json = export_flow_json(tree)

    # Validate structure
    assert "arxiv_id" in flow_json, "Missing arxiv_id in flow JSON"
    assert "children" in flow_json, "Missing children in flow JSON"
    assert "depth" in flow_json,    "Missing depth in flow JSON"

    # Save to disk so river.html can load it
    out_path = REPO_ROOT / "kg" / "visualization" / "flow_data.json"
    with open(out_path, "w") as f:
        json.dump(flow_json, f, indent=2, default=str)

    print(f"✅ Flow JSON exported → {out_path}")
    print(f"   root: {flow_json['arxiv_id']}")
    print(f"   children: {len(flow_json['children'])}")
    print(f"\nOpen kg/visualization/river.html in a browser to see the Research River")

In [ ]:
# ── 5-F: visualization files exist ───────────────────────────────────────────
vis_dir = REPO_ROOT / "kg" / "visualization"

for fname in ["graph.html", "river.html"]:
    fpath = vis_dir / fname
    if fpath.exists():
        size_kb = fpath.stat().st_size // 1024
        print(f"✅ {fname} exists ({size_kb} KB)")
    else:
        print(f"❌ {fname} MISSING — expected at {fpath}")

---
## 6 — CLI Smoke Test
Runs every `kg` command and checks exit codes. No output parsing — just confirms commands don't crash.

In [ ]:
# ── 6-A: install package in editable mode if needed ──────────────────────────
import subprocess, sys

result = subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "-q"],
                        capture_output=True, text=True, cwd=REPO_ROOT)
if result.returncode == 0:
    print("✅ Package installed in editable mode")
else:
    print(f"❌ pip install failed:\n{result.stderr}")

In [ ]:
# ── 6-B: CLI command runner helper ───────────────────────────────────────────
def run_cmd(cmd: list, label: str = None, expect_fail: bool = False):
    """Run a CLI command, print result, return (returncode, stdout)."""
    label = label or " ".join(cmd)
    result = subprocess.run(
        [sys.executable, "-m", "kg.cli"] + cmd,
        capture_output=True, text=True, cwd=REPO_ROOT
    )
    ok = (result.returncode == 0) != expect_fail
    icon = "✅" if ok else "❌"
    print(f"{icon} kg {' '.join(cmd):<35}  exit={result.returncode}")
    if not ok or result.stderr.strip():
        if result.stderr.strip():
            print(f"   stderr: {result.stderr.strip()[:200]}")
    return result.returncode, result.stdout

print("CLI runner ready")

In [ ]:
# ── 6-C: core commands ───────────────────────────────────────────────────────
run_cmd(["status"],                       label="graph status")
run_cmd(["search", "transformer"],        label="keyword search")
run_cmd(["top", "--n", "5"],              label="top papers by rank_score")
run_cmd(["trends"],                       label="trending topics")

In [ ]:
# ── 6-D: enrichment commands ──────────────────────────────────────────────────
run_cmd(["enrich", "--limit", "2", "--dry-run"],  label="enrich dry-run")

# Only test cited-by if we have CITES edges
if not SKIP_FLOW and FLOW_TARGET:
    run_cmd(["cited-by", FLOW_TARGET],  label=f"cited-by {FLOW_TARGET}")
else:
    print("⚠️  cited-by skipped — no CITES edges")

In [ ]:
# ── 6-E: embedding commands ───────────────────────────────────────────────────
run_cmd(["embed", "search", "transformers for vision"],  label="semantic search")

if not SKIP_FLOW and FLOW_TARGET:
    run_cmd(["embed", "similar", FLOW_TARGET],  label=f"similar papers")
else:
    print("⚠️  similar skipped — no flow target")

In [ ]:
# ── 6-F: citation flow commands ───────────────────────────────────────────────
if not SKIP_FLOW and FLOW_TARGET:
    run_cmd(["trace", FLOW_TARGET],            label="trace (depth=2)")
    run_cmd(["trace", FLOW_TARGET, "--depth", "1"], label="trace (depth=1)")
else:
    print("⚠️  trace / flow skipped — no CITES edges")

In [ ]:
# ── 6-G: cluster command (only if we have enough embeddings) ─────────────────
if not SKIP_CLUSTERING:
    run_cmd(["cluster", "run", "--dry-run"],   label="cluster dry-run")
else:
    print("⚠️  cluster skipped — not enough embeddings")

---
## 7 — Full Graph Status Summary
Final check — shows exactly what's in the graph after all tests.

In [ ]:
# ── 7: full graph inventory ───────────────────────────────────────────────────
db = Neo4jClient()
db.connect()

counts = {
    "Papers":                db.run_query("MATCH (p:Paper) RETURN count(p) AS n")[0]["n"],
    "Authors":               db.run_query("MATCH (a:Author) RETURN count(a) AS n")[0]["n"],
    "Topics":                db.run_query("MATCH (t:Topic) RETURN count(t) AS n")[0]["n"],
    "Institutions":          db.run_query("MATCH (i:Institution) RETURN count(i) AS n")[0]["n"],
    "Papers w/ embedding":   db.run_query("MATCH (p:Paper) WHERE p.embedding IS NOT NULL RETURN count(p) AS n")[0]["n"],
    "Papers w/ rank_score":  db.run_query("MATCH (p:Paper) WHERE p.rank_score IS NOT NULL RETURN count(p) AS n")[0]["n"],
    "Papers w/ cluster_id": db.run_query("MATCH (p:Paper) WHERE p.cluster_id IS NOT NULL RETURN count(p) AS n")[0]["n"],
    "WRITTEN_BY edges":      db.run_query("MATCH ()-[:WRITTEN_BY]->() RETURN count(*) AS n")[0]["n"],
    "CITES edges":           db.run_query("MATCH ()-[:CITES]->() RETURN count(*) AS n")[0]["n"],
    "BELONGS_TO edges":      db.run_query("MATCH ()-[:BELONGS_TO]->() RETURN count(*) AS n")[0]["n"],
    "AFFILIATED_WITH edges": db.run_query("MATCH ()-[:AFFILIATED_WITH]->() RETURN count(*) AS n")[0]["n"],
}

db.close()

print("\n" + "═"*45)
print(" GRAPH STATUS AFTER ALL TESTS")
print("═"*45)
for label, count in counts.items():
    bar = "█" * min(count // max(1, max(counts.values()) // 20), 20)
    print(f"  {label:<28} {count:>6}  {bar}")
print("═"*45)
print()

# Expected minimums after running all phases
checks = [
    ("Papers exist",              counts["Papers"] > 0),
    ("Authors exist",             counts["Authors"] > 0),
    ("WRITTEN_BY edges exist",    counts["WRITTEN_BY edges"] > 0),
    ("Enrichment ran",            counts["Papers w/ rank_score"] > 0),
    ("Embeddings stored",         counts["Papers w/ embedding"] > 0),
]

all_pass = True
for label, ok in checks:
    icon = "✅" if ok else "❌"
    print(f"  {icon} {label}")
    if not ok:
        all_pass = False

print()
if all_pass:
    print("🎉 All required checks passed — platform is working!")
else:
    print("⚠️  Some checks failed — review the cells above for details")